# Capstone: Which pages should a content team review first?

Lane 2, Refresh / Content Opportunity Scoring, on the FlyRank pseudonymized warehouse release (build `v20260703`). This notebook is the single reproducible source for the deployed research paper: every number and chart on the page is produced here, and section 7 writes them to `docs/paper/results.json`, which the page reads.

It pulls together weeks 1 to 6: the question (w01, w02), the data contract (w03), the frozen rule baseline (w04), the model (w05) and the honest validation (w06). Seed 42 throughout. June 2026 is the sealed test month, scored once with the final configuration.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "work" / "scripts" / "warehouse.py").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "work" / "scripts"))
import warehouse as W  # shared loader + Lane 2 frame builder (weeks 3-6)

con = W.connect()

import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split

SEED = 42
PAPER_DIR = ROOT / "docs" / "paper"
OUT_DIR = ROOT / "work" / "outputs"
PAPER_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Question

**Research question.** Using only what is known at the start of a month, which pages should a content reviewer look at first, because they are about to lose search visibility faster than the rest of their own site?

**The decision it supports.** A content team can review a few dozen pages per cycle, out of tens of thousands with real search demand. The output is a ranked review queue: each page gets a score, a suggested action (check indexing, refresh, rewrite title and meta, or monitor), and the reason codes behind it. A wrong pick wastes a reviewer's time; a missed pick costs the clicks a page keeps losing until someone notices, and that cost grows with the page's demand. So the queue is judged at the top (precision@K) and by how much of the lost demand it points at, never by accuracy over every page.

**Unit of analysis.** One page at one monthly decision date. Features come only from the calendar month before the decision date; the outcome comes only from the month after it. There are four decision dates with a measured outcome (1 March, 1 April, 1 May, 1 June 2026) and one forward decision (1 July 2026) whose outcome isn't in the release.

In [2]:
MONTHS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05"), ("2026-05", "2026-06")]
frames = {}
for feature_month, label_month in MONTHS:
    d = W.build_frame(con, feature_month, label_month)
    d["tier"] = pd.cut(d["avg_position"], [0, 3, 10, 20, 50, np.inf], labels=False, include_lowest=True)
    frames[label_month] = d

overview = pd.DataFrame({
    f"decision {m}-01": {"feature month": d["feature_month"].iloc[0], "pages": len(d),
                         "clients": d["client_hash_id"].nunique(), "declined (base rate)": round(d["label"].mean(), 3)}
    for m, d in frames.items()
}).T
overview

,feature month,pages,clients,declined (base rate)
decision 2026-03-01,2026-02,57656,24,0.257
decision 2026-04-01,2026-03,84347,36,0.27
decision 2026-05-01,2026-04,98956,41,0.289
decision 2026-06-01,2026-05,105270,40,0.263


## 2. Data

**Release.** `FlyRank/internship-warehouse` on Hugging Face, build `flyrank_pseudonymized_warehouse_release_v20260703`, export date 3 July 2026. Every client and page identifier is a salted pseudonym; no names, domains, URLs, titles or raw queries exist in the release, and none appear here.

**Tables.** `fact_content_daily_performance` (one row per day, client and page), monthly partitions February to June 2026, and `dim_content` for each page's creation date.

**Windows.** Feature month M, outcome month M+1, never sharing a day. Search Console rows only exist on days a page was shown, so tracking is checked per client: a client qualifies for a decision only if its pages have Search Console rows on every calendar day of both months (week 3).

**Eligibility.** At least 100 impressions in the feature month, created at least 30 days before the decision date.

**Excluded, and why.**

| field(s) | why excluded |
|---|---|
| `content_updated_date`, `last_optimized_date`, `word_count`, `is_deleted` | snapshot values from the July export; for most pages they postdate the decision (80.9% of May pages show an update after the decision date) |
| GA4 session and engagement columns | present on about 4% of daily rows; too sparse to use without biasing the slice toward a few clients |
| `sessions_ai` and AI provider columns | too sparse to rank on, and outside this lane |
| `provider_used`, `model_used` | how a page was produced, not how it performs |
| client and page hashes | grouping and splitting only, never features |
| product flags (`health_score`, `priority_score`, `action_type`) | not in the release; would make the result circular if rebuilt |

In [3]:
excluded_check = {
    "snapshot fields in frame features": False,
    "share of May pages with update date after decision": None,
}
dates = con.sql(f"SELECT content_hash_id, content_updated_date FROM read_parquet('{W.file_path('dim_content.parquet')}')").df()
may = frames["2026-05"][["content_hash_id"]].merge(dates, on="content_hash_id", how="left")
share_after = (pd.to_datetime(may["content_updated_date"]) >= pd.Timestamp("2026-05-01")).mean()
total_pages = sum(len(d) for d in frames.values())
print(f"page-decisions across the four labeled months: {total_pages:,}")
print(f"May pages whose snapshot update date is after 1 May: {share_after:.1%}")

page-decisions across the four labeled months: 346,229
May pages whose snapshot update date is after 1 May: 80.9%


## 3. Methodology

**Label.** A page is labeled *declined* when its impressions per day in the outcome month are more than 20% below the feature month **and** that change is at least 20 percentage points worse than the median change across its own client's eligible pages. The second condition removes site-wide movements (seasonality, tracking changes, site events) that a page-level review can't fix. Week 2 showed that most of a raw "down" label moves with the client.

**Features (seven, all from the feature month or fixed metadata).**

| feature | meaning |
|---|---|
| `log_imp` | log of feature-month impressions |
| `days_visible` | days in the feature month with at least one impression |
| `avg_position` | impression-weighted average position |
| `momentum` | log per-day impressions, second half of the month minus first half |
| `content_age_days` | days from creation to the decision date |
| `ctr_pct` | clicks / impressions x 100 |
| `ctr_gap` | CTR minus the median CTR of its position tier (top 3, page 1, 11-20, 21-50, 50+), medians fitted on training rows only |

A client-level momentum feature was tried in week 5 and dropped in week 6: it only helped where clients repeat between training and test.

**Baseline (frozen in week 4).** Flag a page when momentum is below -0.1 *and* its CTR is below its tier's median; rank flagged pages by how hard they slid; everything else scores zero. Reason code `sliding_and_under_clicked`. Both inputs passed a bucket check against the outcome first; search volume, the third candidate, failed it (bigger pages decline *less* often).

**Model.** A random forest (300 trees, leaves of at least 50 pages), chosen over logistic regression and gradient boosting in week 5 because it was best on full-list ranking quality and boosting added nothing.

**Validation.** Four designs, the rule scored on the same rows every time: a random 80/20 page split (shown only as the "before"), 5-fold cross-validation grouped by client, time-forward (train on the March and April decisions, test on May, with unseen clients reported separately), and the sealed June month (train on March to May, test on June, run once).

**Leakage checks.** Checked as assertions below: windows in order, no label-side column among features, no snapshot field or product flag, tier medians fitted on training rows only. A deliberately leaked column lifted held-out AUC from 0.66 to 0.89 in week 3, which confirmed the harness catches leaks.

In [4]:
FEATURES = W.FIVE_FEATURES + ["ctr_pct", "ctr_gap"]
LABEL_SIDE = {"imp_next", "change_pct", "client_median_change", "vs_client", "label", "lost_per_day"}
SNAPSHOT = {"content_updated_date", "last_optimized_date", "word_count", "char_count", "is_deleted", "is_published"}
PRODUCT_FLAGS = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"}

for m, d in frames.items():
    assert pd.Period(d["feature_month"].iloc[0]) + 1 == pd.Period(m), "windows out of order"
assert not set(FEATURES) & (LABEL_SIDE | SNAPSHOT), "leaky feature"
assert not any(set(d.columns) & PRODUCT_FLAGS for d in frames.values()), "product flag present"

def add_ctr_gap(train, *others):
    med = train.groupby("tier")["ctr_pct"].median()
    for d in (train, *others):
        d["ctr_gap"] = d["ctr_pct"] - d["tier"].map(med)

def rule_score(d):
    med = d.groupby("tier")["ctr_pct"].transform("median")
    flagged = d["momentum"].lt(-0.1) & (d["ctr_pct"] - med).lt(0)
    return np.where(flagged, -d["momentum"], 0.0)

def forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED)

def metrics(y, s, lost):
    y, s = np.asarray(y), np.asarray(s)
    return {"ROC AUC": roc_auc_score(y, s), "avg precision": average_precision_score(y, s),
            "P@50": W.precision_at_k(s, y, 50), "P@500": W.precision_at_k(s, y, 500),
            "demand@50": W.demand_captured_at_k(s, y, lost, 50), "base rate": y.mean(), "rows": len(y)}
print("leakage assertions passed")

leakage assertions passed


In [5]:
# Signal checks behind the rule (April decision, as in week 4).
april = frames["2026-04"].copy()
april["ctr_gap_global"] = april["ctr_pct"] - april.groupby("tier")["ctr_pct"].transform("median")
signals = {}
for name, col, labels in [
    ("momentum", "momentum", ["sliding hard", "sliding", "flat", "rising", "rising hard"]),
    ("ctr_vs_tier", "ctr_gap_global", ["far below", "below", "near", "above", "far above"]),
]:
    b = pd.qcut(april[col].rank(method="first"), 5, labels=labels)
    t = april.groupby(b, observed=True)["label"].agg(n="size", rate="mean")
    signals[name] = [{"bucket": str(k), "n": int(v["n"]), "rate": round(float(v["rate"]), 4)} for k, v in t.iterrows()]
vol = pd.cut(april["imp"], [100, 300, 1_000, 3_000, 10_000, np.inf], right=False,
             labels=["100-300", "300-1K", "1K-3K", "3K-10K", "10K+"])
t = april.groupby(vol, observed=True)["label"].agg(n="size", rate="mean")
signals["volume"] = [{"bucket": str(k), "n": int(v["n"]), "rate": round(float(v["rate"]), 4)} for k, v in t.iterrows()]
signals["base_rate"] = round(float(april["label"].mean()), 4)
pd.DataFrame({k: [f"{r['bucket']}: {r['rate']:.1%} (n={r['n']:,})" for r in v] for k, v in signals.items() if k != "base_rate"})

,momentum,ctr_vs_tier,volume
0,"sliding hard: 51.9% (n=16,870)","far below: 39.5% (n=16,870)","100-300: 31.2% (n=22,638)"
1,"sliding: 28.0% (n=16,869)","below: 30.5% (n=16,869)","300-1K: 29.9% (n=23,539)"
2,"flat: 20.0% (n=16,869)","near: 27.9% (n=16,869)","1K-3K: 25.6% (n=18,510)"
3,"rising: 17.4% (n=16,869)","above: 20.5% (n=16,869)","3K-10K: 20.2% (n=14,276)"
4,"rising hard: 17.7% (n=16,870)","far above: 16.4% (n=16,870)","10K+: 19.2% (n=5,384)"


## 4. Results (vs baseline)

Every row below scores the forest and the frozen rule on exactly the same pages.

In [6]:
rows = {}

# Before: random 80/20 over April pages.
tr, te = train_test_split(np.arange(len(april)), test_size=0.2, random_state=SEED)
A, B = april.iloc[tr].copy(), april.iloc[te].copy()
add_ctr_gap(A, B)
s = forest().fit(A[FEATURES], A["label"]).predict_proba(B[FEATURES])[:, 1]
rows[("random 80/20 (before)", "forest")] = metrics(B["label"], s, B["lost_per_day"])
rows[("random 80/20 (before)", "rule")] = metrics(B["label"], rule_score(april)[te], B["lost_per_day"])

# Grouped by client, April.
oof = np.zeros(len(april))
for tr, te in GroupKFold(n_splits=5).split(april, april["label"], april["client_hash_id"]):
    A, B = april.iloc[tr].copy(), april.iloc[te].copy()
    add_ctr_gap(A, B)
    oof[te] = forest().fit(A[FEATURES], A["label"]).predict_proba(B[FEATURES])[:, 1]
rows[("grouped by client", "forest")] = metrics(april["label"], oof, april["lost_per_day"])
rows[("grouped by client", "rule")] = metrics(april["label"], rule_score(april), april["lost_per_day"])

# Time-forward to May.
train = pd.concat([frames["2026-03"], frames["2026-04"]]).copy()
may = frames["2026-05"].copy()
add_ctr_gap(train, may)
s = forest().fit(train[FEATURES], train["label"]).predict_proba(may[FEATURES])[:, 1]
rows[("time-forward to May", "forest")] = metrics(may["label"], s, may["lost_per_day"])
rows[("time-forward to May", "rule")] = metrics(may["label"], rule_score(may), may["lost_per_day"])

# Sealed June: final configuration, run once.
train = pd.concat([frames["2026-03"], frames["2026-04"], frames["2026-05"]]).copy()
june = frames["2026-06"].copy()
add_ctr_gap(train, june)
final_model = forest().fit(train[FEATURES], train["label"])
june["forest"] = final_model.predict_proba(june[FEATURES])[:, 1]
june["rule"] = rule_score(june)
rows[("sealed June", "forest")] = metrics(june["label"], june["forest"], june["lost_per_day"])
rows[("sealed June", "rule")] = metrics(june["label"], june["rule"], june["lost_per_day"])

results = pd.DataFrame(rows).T
results["rows"] = results["rows"].astype(int)
results.index.names = ["design", "method"]
results.round(3)

ROC AUC  avg precision  P@50  P@500  demand@50  base rate    rows
design                method                                                                   
random 80/20 (before) forest    0.748          0.551  0.96  0.818      0.015      0.271   16870
                      rule      0.593          0.393  0.96  0.774      0.034      0.271   16870
grouped by client     forest    0.707          0.488  0.90  0.832      0.013      0.270   84347
                      rule      0.595          0.389  0.96  0.874      0.069      0.270   84347
time-forward to May   forest    0.729          0.583  0.96  0.938      0.017      0.289   98956
                      rule      0.634          0.471  0.98  0.920      0.137      0.289   98956
sealed June           forest    0.703          0.496  0.84  0.810      0.034      0.263  105270
                      rule      0.602          0.399  1.00  1.000      0.085      0.263  105270

In [7]:
# One client dominates the rule's June top 500; report June with and without it.
top_client = june.sort_values("rule", ascending=False).head(500)["client_hash_id"].value_counts()
event_client = top_client.index[0]
print(f"rule's June top 500: {top_client.iloc[0]} pages from one client")
rest = june[june["client_hash_id"] != event_client].copy()

june["demand"] = june["imp"] / 31
rest["demand"] = rest["imp"] / 31
tradeoff = {}
for label, d in [("all June pages", june), ("June without the event client", rest)]:
    for name, s in [("forest", d["forest"]), ("rule", d["rule"]),
                    ("forest x demand", d["forest"] * d["demand"]), ("rule x demand", d["rule"] * d["demand"])]:
        m = metrics(d["label"], s, d["lost_per_day"])
        tradeoff[(label, name)] = {"P@50": m["P@50"], "P@500": m["P@500"], "demand@50": m["demand@50"]}
tradeoff = pd.DataFrame(tradeoff).T
tradeoff.round(3)

rule's June top 500: 495 pages from one client


P@50  P@500  demand@50
all June pages                forest           0.84  0.810      0.034
                              rule             1.00  1.000      0.085
                              forest x demand  0.48  0.430      0.604
                              rule x demand    0.78  0.924      0.514
June without the event client forest           0.82  0.814      0.031
                              rule             0.82  0.804      0.087
                              forest x demand  0.48  0.432      0.604
                              rule x demand    0.76  0.590      0.509

In [8]:
# Precision at K on sealed June (without the event client), for the chart.
ks = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
pk = {"k": ks, "base_rate": round(float(rest["label"].mean()), 4)}
for name in ["forest", "rule"]:
    pk[name] = [round(W.precision_at_k(rest[name], rest["label"], k), 4) for k in ks]
rng = np.random.default_rng(SEED)
pk["random"] = [round(W.precision_at_k(rng.random(len(rest)), rest["label"], k), 4) for k in ks]
pd.DataFrame(pk).drop(columns="base_rate").set_index("k").T

k,10,20,50,100,200,500,1000,2000,5000
forest,0.8,0.80,0.82,0.82,0.830,0.814,0.787,0.7425,0.6726
rule,0.9,0.90,0.82,0.84,0.835,0.804,0.771,0.7280,0.6420
random,0.1,0.45,0.14,0.25,0.240,0.290,0.263,0.2490,0.2510


In [9]:
# What the final model leans on, measured on sealed June.
sample = june.sample(20_000, random_state=SEED)
imp = permutation_importance(final_model, sample[FEATURES], sample["label"], scoring="average_precision",
                             n_repeats=5, random_state=SEED, n_jobs=1)
importance = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
importance.round(4).to_frame("AP drop when shuffled (June)")

,AP drop when shuffled (June)
momentum,0.1932
days_visible,0.0127
ctr_gap,0.0126
log_imp,0.0086
ctr_pct,0.0075
avg_position,-0.0030
content_age_days,-0.0054


Four things the results show.

**Honest splits matter.** The random page split credits the forest with 0.748 ROC AUC; grouped by client it's 0.707. The rule doesn't move (0.593 vs 0.595) because it learns nothing from training data, so the gap is client behaviour the forest memorised, not a harder test.

**The forest ranks the whole list better, every time.** ROC AUC 0.70 to 0.73 against the rule's 0.60 to 0.63, and average precision about 0.10 higher, in every honest design, including the sealed June month (0.703 vs 0.602).

**At the top, the two are level.** The rule's perfect June score is one client: 495 of its top 500 pages. Without that client both reach 0.82 at 50 and 0.80 to 0.81 at 500. The precision@K curve has them tracking each other up to about 500 and the forest pulling ahead after 1,000 (0.673 vs 0.642 at 5,000).

**Demand weighting is a real trade-off.** Forest x demand points at 0.604 of June's lost demand in its top 50, against about 0.51 for rule x demand, at precision 0.48 against 0.76 to 0.78.

Momentum carries the model: shuffling it drops June average precision by 0.19. Days visible and the CTR gap add about 0.01 each; position and content age add nothing measurable on June.

## 5. Limitations

What this work can say: on four monthly decisions from one pseudonymized release, a seven-feature random forest was measured ahead of a transparent rule on full-list ranking quality under client-grouped, time-forward and sealed-month validation, and the two were roughly level for the top 50 to 500 pages a team could review. That's observed, directional, decision-support evidence.

What it can't say:

- **Nothing causal.** The data says which pages declined, not whether a refresh would have stopped it. Testing that needs an experiment (refresh a random half of the queue, compare).
- **Not about Google's algorithm.** These are associations in one portfolio of 24 to 41 tracked clients a month.
- **The label is mine.** "More than 20% down and 20 points worse than the client" is a defensible choice, not a fact; a different threshold changes the base rate and the numbers.
- **One month of features.** Large, steady pages that collapse with no warning are the biggest misses, and a one-month window can't see them coming.
- **Client concentration.** One client can dominate a month (495 of the rule's June top 500), so per-client caps and site-level checks are part of using the queue, not optional extras.
- **Data quality.** Some pages report an average position below 1, which Search Console can't produce (485 in April, 15 in June); they're flagged, not trusted.
- **Staleness is unmeasurable at decision time** in this release, so the signal behind the classic refresh flag isn't tested here.

In [10]:
quirk = (june["avg_position"] < 1)
print(f"June pages with average position below 1: {quirk.sum():,} ({quirk.mean():.2%})")
print(f"clients per labeled month: {min(d['client_hash_id'].nunique() for d in frames.values())} to "
      f"{max(d['client_hash_id'].nunique() for d in frames.values())}")
print(f"share of June declines outside the forest's top 5,000: "
      f"{1 - june.nlargest(5000, 'forest')['label'].sum() / june['label'].sum():.1%}")

June pages with average position below 1: 15 (0.01%)
clients per labeled month: 24 to 41
share of June declines outside the forest's top 5,000: 86.6%


## 6. Ranked recommendations

The forward queue: the final model retrained on all four labeled decisions (March to June), scoring pages for the **1 July 2026** decision from their June features. No outcome exists for July in the release, so this is the queue a team would receive, not a result.

Each page gets one action from readable checks (first match wins) and every reason code that applies:

| order | action | when |
|---|---|---|
| 1 | `check_indexing` | visible on 15 or fewer days of June, or momentum at or below -1.5 (it dropped out of search late in the month) |
| 2 | `refresh_and_rewrite_snippet` | sliding (momentum below -0.1) and CTR below its tier |
| 3 | `refresh_content` | sliding, CTR at or above its tier |
| 4 | `rewrite_snippet` | CTR below its tier on page 1 or better, not sliding |
| 5 | `monitor` | none of the above |

Ordering: expected lost demand (forest probability x June impressions per day), capped at 10 pages per client in the top 50 so one site can't fill the queue. Pages with an impossible position get the `position_data_suspect` code so a reviewer checks tracking first.

In [11]:
full_train = pd.concat([frames[m] for m in ["2026-03", "2026-04", "2026-05", "2026-06"]]).copy()
july = W.build_forward_features(con, "2026-06")
july["tier"] = pd.cut(july["avg_position"], [0, 3, 10, 20, 50, np.inf], labels=False, include_lowest=True)
add_ctr_gap(full_train, july)
queue_model = forest().fit(full_train[FEATURES], full_train["label"])
july["p_decline"] = queue_model.predict_proba(july[FEATURES])[:, 1]
july["demand_per_day"] = july["imp"] / 30
july["expected_loss"] = july["p_decline"] * july["demand_per_day"]

def action(r):
    if r["days_visible"] <= 15 or r["momentum"] <= -1.5:
        return "check_indexing"
    if r["momentum"] < -0.1 and r["ctr_gap"] < 0:
        return "refresh_and_rewrite_snippet"
    if r["momentum"] < -0.1:
        return "refresh_content"
    if r["ctr_gap"] < 0 and r["avg_position"] <= 10:
        return "rewrite_snippet"
    return "monitor"

def reasons(r):
    codes = []
    if r["days_visible"] <= 15 or r["momentum"] <= -1.5: codes.append("dropped_out_late_month")
    elif r["momentum"] < -0.1: codes.append("sliding")
    if r["ctr_gap"] < 0: codes.append("under_clicked_for_position")
    if r["imp"] >= 3000: codes.append("high_demand")
    if r["avg_position"] < 1: codes.append("position_data_suspect")
    return ";".join(codes) or "none"

july["action"] = july.apply(action, axis=1)
july["reason_codes"] = july.apply(reasons, axis=1)

ranked = july.sort_values("expected_loss", ascending=False).copy()
ranked["client_rank"] = ranked.groupby("client_hash_id").cumcount() + 1
capped_top = ranked[ranked["client_rank"] <= 10].head(50)
queue = pd.concat([capped_top, ranked.drop(capped_top.index)])
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue[["rank", "content_hash_id", "client_hash_id", "p_decline", "expected_loss", "action", "reason_codes",
       "imp", "days_visible", "avg_position", "momentum", "ctr_pct"]].to_csv(OUT_DIR / "capstone_july_queue.csv", index=False)
print(f"July queue: {len(queue):,} pages from {queue['client_hash_id'].nunique()} clients -> work/outputs/capstone_july_queue.csv")
print(f"clients in the top 50: {capped_top['client_hash_id'].nunique()}, max pages from one client: {capped_top['client_hash_id'].value_counts().max()}")
mix = capped_top["action"].value_counts().rename("top 50").to_frame()
mix["all pages"] = queue["action"].value_counts()
mix.fillna(0).astype(int)

July queue: 96,727 pages from 43 clients -> work/outputs/capstone_july_queue.csv
clients in the top 50: 12, max pages from one client: 10


,top 50,all pages
action,,
refresh_and_rewrite_snippet,25,12218
refresh_content,14,38688
check_indexing,7,3211
rewrite_snippet,2,5779
monitor,2,36831


In [12]:
top10 = capped_top.head(10).copy()
top10["page"] = top10["content_hash_id"].str.slice(0, 16) + "..."
top10[["page", "p_decline", "expected_loss", "action", "reason_codes", "imp", "days_visible", "avg_position", "momentum"]].round(3)

,page,p_decline,expected_loss,action,reason_codes,imp,days_visible,avg_position,momentum
56755,content_963de14b...,0.471,9653.137,refresh_content,sliding;high_demand,615012.0,30,6.399,-0.750
55961,content_943dc881...,0.769,7491.578,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,292416.0,30,1.927,-0.933
69951,content_b9023208...,0.699,5472.493,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,234902.0,30,5.700,-1.069
77105,content_cc26620b...,0.583,4382.923,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,225417.0,30,6.000,-1.015
78747,content_d0acf706...,0.893,4316.676,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,145022.0,30,2.538,-1.135
19414,content_33d31496...,0.578,3261.666,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,169233.0,30,7.479,-0.581
73331,content_c1f764a2...,0.412,3236.139,refresh_and_rewrite_snippet,sliding;under_clicked_for_position;high_demand,235427.0,30,7.074,-0.425
6611,content_11bf4c33...,0.855,3055.872,check_indexing,dropped_out_late_month;under_clicked_for_position;high_demand,107215.0,30,8.896,-2.455
60121,content_9ef3d751...,0.329,2964.630,refresh_content,sliding;high_demand,269943.0,30,2.075,-0.442
39534,content_68ffc95c...,0.861,2805.373,check_indexing,dropped_out_late_month;under_clicked_for_position;high_demand,97740.0,30,10.151,-3.094


The playbook, in the order a reviewer should work it:

1. **Indexing checks first.** 7 of the top 50 (3,211 pages overall) dropped out of search late in June. A status check (noindex, 404, redirect, merge) takes minutes, and in every labeled month pages like these almost always stayed down. Fix tracking or status before touching content.
2. **Then the demand-ranked refresh queue.** 25 of the top 50 are sliding *and* under-clicked for their position: refresh the content and rewrite the title and meta. 14 more are sliding with healthy CTR: refresh the content, leave the snippet. These are very large pages (the first has 615K June impressions), so even a moderate decline probability puts a lot of demand at risk.
3. **Snippet-only rewrites** where a page-1 page is under-clicked but not sliding (2 in the top 50).
4. **When a client hits the 10-page cap, look at the site first.** Ten sliding pages from one site in one month is more likely a site event than ten content problems.
5. **Log every action and re-score next month.** The July outcome becomes the next label. Refreshing a random half of the queue and leaving the other half untouched is the only way to learn whether the refreshes themselves help; this paper can't claim that.

## 7. Artifacts the paper embeds

Everything the paper shows is written to `docs/paper/results.json` (aggregates and pseudonymized, truncated page ids only), plus the same data as `results.js` for the page to load. The page draws its charts from it, so re-running this notebook updates the paper. A preview of the main charts follows.

In [13]:
def rec(df):
    return [{**{k: (v if isinstance(v, str) else round(float(v), 4)) for k, v in r.items()}} for r in df]

results_out = results.reset_index()
payload = {
    "release": "flyrank_pseudonymized_warehouse_release_v20260703",
    "decisions": [{"decision": f"{m}-01", "feature_month": d["feature_month"].iloc[0], "pages": int(len(d)),
                   "clients": int(d["client_hash_id"].nunique()), "base_rate": round(float(d["label"].mean()), 4)}
                  for m, d in frames.items()],
    "total_page_decisions": int(total_pages),
    "signals": signals,
    "validation": rec(results_out.to_dict(orient="records")),
    "tradeoff": rec(tradeoff.reset_index().rename(columns={"level_0": "slice", "level_1": "method"}).to_dict(orient="records")),
    "precision_at_k_june_excl_event": pk,
    "importance": [{"feature": k, "ap_drop": round(float(v), 4)} for k, v in importance.items()],
    "event_client_share_of_rule_top500": int(top_client.iloc[0]),
    "july_queue": {
        "pages": int(len(queue)), "clients": int(queue["client_hash_id"].nunique()),
        "action_mix_top50": {k: int(v) for k, v in capped_top["action"].value_counts().items()},
        "action_mix_all": {k: int(v) for k, v in queue["action"].value_counts().items()},
        "top10": [{"rank": int(r["rank"]), "page": r["content_hash_id"][:16] + "...", "p_decline": round(float(r["p_decline"]), 3),
                   "expected_loss": round(float(r["expected_loss"]), 1), "action": r["action"], "reasons": r["reason_codes"],
                   "impressions": int(r["imp"]), "days_visible": int(r["days_visible"]), "avg_position": round(float(r["avg_position"]), 1)}
                  for _, r in queue.head(10).iterrows()],
    },
}
(PAPER_DIR / "results.json").write_text(json.dumps(payload, indent=1))
(PAPER_DIR / "results.js").write_text("window.RESULTS = " + json.dumps(payload) + ";" + chr(10))
(OUT_DIR / "capstone_metrics.json").write_text(json.dumps({"validation": payload["validation"], "tradeoff": payload["tradeoff"]}, indent=1))
print("wrote docs/paper/results.json, docs/paper/results.js and work/outputs/capstone_metrics.json")

wrote docs/paper/results.json, docs/paper/results.js and work/outputs/capstone_metrics.json


In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
names = [r["bucket"] for r in signals["momentum"]]
ax.bar(names, [r["rate"] for r in signals["momentum"]], color=BLUE, width=0.6)
ax.axhline(signals["base_rate"], color=GRAY, lw=1)
ax.set_title("Decline rate by late-month momentum (April)", fontsize=10, loc="left")
ax.set_ylabel("share of pages that declined")

ax = axes[1]
designs = results_out["design"].unique()
x = np.arange(len(designs))
for i, (m, c) in enumerate([("forest", BLUE), ("rule", ORANGE)]):
    vals = [results_out[(results_out["design"] == d) & (results_out["method"] == m)]["ROC AUC"].iloc[0] for d in designs]
    ax.bar(x + (i - 0.5) * 0.36, vals, width=0.34, color=c, label=m)
ax.set_xticks(x, designs, fontsize=8)
ax.set_ylim(0.5, 0.8)
ax.set_title("ROC AUC by validation design", fontsize=10, loc="left")
ax.legend(frameon=False)

ax = axes[2]
for name, c in [("forest", BLUE), ("rule", ORANGE), ("random", GRAY)]:
    ax.plot(pk["k"], pk[name], color=c, lw=2, marker="o", ms=4, label=name)
ax.set_xscale("log")
ax.set_title("Precision@K, sealed June (event client excluded)", fontsize=10, loc="left")
ax.set_xlabel("K (pages reviewed)")
ax.legend(frameon=False)
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
display(fig)
plt.close(fig)

<Figure size 1500x420 with 3 Axes>

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (seed 42; June scored once with the final configuration)
- [x] No client names, URLs, or private queries anywhere (pseudonymized hashes, truncated in the paper)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed under `work/notebooks/`; paper URL in `submission/paper_url.txt`